# 14. Integrated_Validator_Node.py 테스트

이 노트북은 `src.Integrated_Validator_Node` 모듈의 `IntegratedValidator` 클래스를 테스트하기 위해 작성되었습니다.
- 1계층: `NarrativeConstraintSolver`를 활용한 구조적(ID) 전이 검증
- 2계층: 캐릭터 금기 행동(`Forbidden_Action`) 기반 대화 맥락 검증
- 동적 수정: 전환점(`is_turning_point`) 시 금기 행동의 '트라우마 극복' 합법화(Legalization) 로직 검증

In [1]:
import os
import sys
import json

# 프로젝트 루트를 sys.path에 추가 (src 임포트 가능하게 함)
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

from src.Integrated_Validator_Node import IntegratedValidator
print("✅ IntegratedValidator 임포트 성공")

✅ IntegratedValidator 임포트 성공


## 1. 테스트 데이터 준비
`IntegratedValidator` 초기화에 필요한 `schema_data.json` 경로를 확인하고 인스턴스를 생성합니다.

In [ ]:
schema_path = "../data/schema_data_modified.json"
validator = IntegratedValidator(theory_type="propp", schema_path=schema_path)

print(f"✅ Validator 초기화 완료 (Schema: {schema_path})")

JSONDecodeError: Expecting property name enclosed in double quotes: line 2 column 59 (char 60)

## 2. 시나리오 테스트 (성공 케이스)
전이 경로가 올바르고 금기 행동이 없는 경우입니다.

In [ ]:
test_state = {
    "generated_proposals": [
        {"func_id": "PROPP_01_ABSENTATION", "content": "부모님이 집을 비웠다."},
        {"func_id": "PROPP_02_INTERDICTION", "content": "아이에게 숲속으로 가지 말라고 경고했다."}
    ],
    "archetype_locks": {
        "char_hero": {"Forbidden_Action": ["거짓말하기", "도망치기"]}
    },
    "violation_log": [],
    "pending_queries": [],
    "is_turning_point": False
}

result = validator.validate_node(test_state)
print("🔍 검증 결과:")
print(json.dumps(result, indent=2, ensure_ascii=False))

## 3. 시나리오 테스트 (실패 케이스: 금기 행동 감지)
캐릭터의 금기 행동이 포함된 경우입니다.

In [ ]:
test_state_fail = {
    "generated_proposals": [
        {"func_id": "PROPP_01_ABSENTATION", "content": "부모님이 집을 비웠다."},
        {"func_id": "PROPP_02_INTERDICTION", "content": "철수는 무서워서 숲에서 도망치기 시작했다."}
    ],
    "archetype_locks": {
        "char_hero": {"Forbidden_Action": ["도망치기"]}
    },
    "violation_log": ["이전 로그 1"],
    "pending_queries": [],
    "is_turning_point": False
}

result_fail = validator.validate_node(test_state_fail)
print("🔍 검증 결과 (금기 행동 발생):")
print(json.dumps(result_fail, indent=2, ensure_ascii=False))

## 4. 시나리오 테스트 (동적 합법화: 전환점에서의 극복)
`is_turning_point=True`가 설정되면 금기 행동도 통과됩니다.

In [ ]:
test_state_turning = {
    "generated_proposals": [
        {"func_id": "PROPP_01_ABSENTATION", "content": "부모님이 집을 비웠다."},
        {"func_id": "PROPP_02_INTERDICTION", "content": "철수는 이제 도망치기 보다는 맞서 싸우기로 했다."}
    ],
    "archetype_locks": {
        "char_hero": {"Forbidden_Action": ["도망치기"]}
    },
    "violation_log": [],
    "pending_queries": [],
    "is_turning_point": True
}

result_turning = validator.validate_node(test_state_turning)
print("🔍 검증 결과 (전환점 극복):")
print(json.dumps(result_turning, indent=2, ensure_ascii=False))